In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

import warnings
warnings.filterwarnings("ignore")

### Book Recommendation System

In [2]:
dataset = pd.read_csv("book_dataset.csv")
dataset.sample(2)

,bookId,title,series,author,rating,description,language,isbn,genres,characters,...,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price
40998,8105542-invincible-summer,Invincible Summer,NaN,Hannah Moskowitz (Goodreads Author),3.57,Noah’s happier than I’ve seen him in months. S...,English,9781442407510,"['Young Adult', 'Contemporary', 'Romance', 'Fi...",[],...,NaN,[],2201,"['649', '565', '547', '265', '175']",80.0,[],https://i.gr-assets.com/images/S/compressed.ph...,80,1,3.12
42587,16123002-dark-star,Dark Star,The Rosie Black Chronicles #3,Lara Morgan (Goodreads Author),4.03,To protect Pip and fulfil her deal with Sulawa...,English,9781921529412,"['Science Fiction', 'Dystopia', 'Young Adult',...",[],...,NaN,[],315,"['114', '115', '71', '11', '4']",95.0,[],https://i.gr-assets.com/images/S/compressed.ph...,76,1,NaN


In [3]:
dataset.columns, dataset.shape

(Index(['bookId', 'title', 'series', 'author', 'rating', 'description',
        'language', 'isbn', 'genres', 'characters', 'bookFormat', 'edition',
        'pages', 'publisher', 'publishDate', 'firstPublishDate', 'awards',
        'numRatings', 'ratingsByStars', 'likedPercent', 'setting', 'coverImg',
        'bbeScore', 'bbeVotes', 'price'],
       dtype='object'),
 (52478, 25))

In [4]:
cleaned_df = dataset.drop(['bookId','isbn', 'characters', 'bookFormat', 'edition', 'publisher', 'publishDate', 'firstPublishDate', 'awards', 'numRatings', 'ratingsByStars', 'likedPercent', 'setting','bbeScore', 'bbeVotes', 'price'], axis=1)

cleaned_df.isna().sum(), cleaned_df.shape

(title              0
 series         29008
 author             0
 rating             0
 description     1338
 language        3806
 genres             0
 pages           2347
 coverImg         605
 dtype: int64,
 (52478, 9))

In [5]:
cleaned_df = cleaned_df[cleaned_df['language'] == 'English']
cleaned_df.drop('language', axis=1, inplace=True)
cleaned_df.dropna(subset=['description'], inplace=True)
cleaned_df['pages'] = cleaned_df['pages'].fillna(300)
# cleaned_df = cleaned_df.fillna(0)
cleaned_df.isna().sum(), cleaned_df.shape
     # , cleaned_df.duplicated().sum())

(title              0
 series         21535
 author             0
 rating             0
 description        0
 genres             0
 pages              0
 coverImg         180
 dtype: int64,
 (42151, 8))

In [6]:
cleaned_df.columns

Index(['title', 'series', 'author', 'rating', 'description', 'genres', 'pages',
       'coverImg'],
      dtype='object')

In [7]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42151 entries, 0 to 52477
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        42151 non-null  object 
 1   series       20616 non-null  object 
 2   author       42151 non-null  object 
 3   rating       42151 non-null  float64
 4   description  42151 non-null  object 
 5   genres       42151 non-null  object 
 6   pages        42151 non-null  object 
 7   coverImg     41971 non-null  object 
dtypes: float64(1), object(7)
memory usage: 2.9+ MB


In [8]:
cleaned_df.head(5)

,title,series,author,rating,description,genres,pages,coverImg
0,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...",374,https://i.gr-assets.com/images/S/compressed.ph...
1,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",870,https://i.gr-assets.com/images/S/compressed.ph...
2,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,"['Classics', 'Fiction', 'Historical Fiction', ...",324,https://i.gr-assets.com/images/S/compressed.ph...
3,Pride and Prejudice,NaN,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,"['Classics', 'Fiction', 'Romance', 'Historical...",279,https://i.gr-assets.com/images/S/compressed.ph...
4,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...",501,https://i.gr-assets.com/images/S/compressed.ph...


In [9]:
cleaned_df['author']

0                                          Suzanne Collins
1                J.K. Rowling, Mary GrandPré (Illustrator)
2                                               Harper Lee
3                Jane Austen, Anna Quindlen (Introduction)
4                                          Stephenie Meyer
                               ...                        
52473                     Cheri Schmidt (Goodreads Author)
52474                                        Emma Michaels
52475                    Kim Richardson (Goodreads Author)
52476    Tom Pollack (Goodreads Author), John Loftus (G...
52477                      Misty Moncur (Goodreads Author)
Name: author, Length: 42151, dtype: object

In [10]:
cleaned_df['author'].value_counts()

author
Nora Roberts (Goodreads Author)                 85
Agatha Christie                                 69
Stephen King (Goodreads Author)                 62
Bella Forrest (Goodreads Author)                49
Meg Cabot (Goodreads Author)                    49
                                                ..
Gail Godwin                                      1
Katharine Eliska Kimbriel (Goodreads Author)     1
Bo Carpelan, David McDuff (translator)           1
Charles Sailor (Goodreads Author)                1
Misty Moncur (Goodreads Author)                  1
Name: count, Length: 22064, dtype: int64

In [11]:
import ast

def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i)
    return L


cleaned_df['genres'] = cleaned_df['genres'].apply(convert)
cleaned_df['genres'] = cleaned_df['genres'].apply(lambda x : ' '.join(x))
cleaned_df['author'] = cleaned_df['author'].fillna('').str.extract(r'^\(?\s*([^,(]+)')

In [12]:
cleaned_df['tags'] = cleaned_df['author'] + cleaned_df['genres'] + cleaned_df['description']
cleaned_df['tags'] = cleaned_df['tags'].str.lower()
cleaned_df.sample(3)

,title,series,author,rating,description,genres,pages,coverImg,tags
19334,These Strange Ashes,NaN,Elisabeth Elliot,4.33,In her first year as a missionary to a small g...,Christian Biography Nonfiction Christian Livin...,152,https://i.gr-assets.com/images/S/compressed.ph...,elisabeth elliotchristian biography nonfiction...
747,To All the Boys I've Loved Before,To All the Boys I've Loved Before #1,Jenny Han,4.15,To All the Boys I’ve Loved Before is the story...,Young Adult Romance Contemporary Fiction Audio...,355,https://i.gr-assets.com/images/S/compressed.ph...,jenny han young adult romance contemporary fic...
29903,Dead Is a Killer Tune,Dead Is #7,Marlene Perez,4.21,High school freshman Jessica Walsh is a Virago...,Young Adult Paranormal Mystery Fantasy Superna...,224,https://i.gr-assets.com/images/S/compressed.ph...,marlene perez young adult paranormal mystery f...


In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stopword = stopwords.words('english')

    for word in tokens:
        if word in stopword:
            text = text.replace(word, "")
    return text

cleaned_df['tags'] = cleaned_df['tags'].apply(remove_stopwords)

In [14]:
def stemming(text):
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_words = []

    for token in tokens:
        stemmed_word = ps.stem(token)
        stemmed_words.append(stemmed_word)

    return " ".join(stemmed_words)

cleaned_df['tags'] = cleaned_df['tags'].apply(stemming)

In [15]:
cleaned_df.sample(3)

,title,series,author,rating,description,genres,pages,coverImg,tags
7111,Fever,Breathless #2,Maya Banks,4.00,"Jace, Ash, and Gabe: three of the wealthiest, ...",Romance Erotica BDSM Contemporary Romance Cont...,416,https://i.gr-assets.com/images/S/compressed.ph...,my bnk romnc eroic bdm cemporri romnc cemporri...
3856,Bunnicula,Bunnicula #1,Deborah Howe,3.86,BEWARE THE HARE!\nIs he or isn't he a vampire?...,Childrens Fiction Fantasy Mystery Horror Anima...,98,https://i.gr-assets.com/images/S/compressed.ph...,debh howechildren fiction fntsi mysteri hr nim...
18773,Bubbles and Billy Sandwalker,NaN,Cyndie Lepori-Mogavero,4.50,Just a fun little good book about the sea and ...,,109,https://i.gr-assets.com/images/S/compressed.ph...,cynd lepor-mogverojust fun lttle good book bou...


In [16]:
tf = TfidfVectorizer(max_features = 5000)
vectors = tf.fit_transform(cleaned_df['tags'])
vectors.shape

(42151, 5000)

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(movie_name):
    book_idx = cleaned_df[cleaned_df['title'] == movie_name].index[0]
    distance = cosine_similarity(vectors[book_idx], vectors).flatten()
    books_list = sorted(list(enumerate(distance)), key=lambda x: x[1], reverse=True)[1:15]

    books = []

    for idx, _ in books_list:
        book_name = cleaned_df.iloc[idx]['title']
        author = cleaned_df.iloc[idx]['author']
        books.append(book_name + ' by ' +  author)
    return books

In [18]:
recommend("The Hunger Games")
# The Hunger Games Suzanne Collins

['The Hunger Games Tribute Guide by Emily Seife ',
 'SAMPLER ONLY: Catching Fire (The Hunger Games, #2) by Suzanne Collins',
 'Mockingjay by Suzanne Collins',
 'Guide to The Hunger Games by Caroline Carpenter',
 'The Hunger Games Trilogy Boxset by Suzanne Collins',
 'Finally by Wendy Mass ',
 'The Book of Atrus by Rand Miller',
 'The Hunger But Mainly Death Games: A Parody by Bratniss Everclean ',
 'The Ballad of Songbirds and Snakes by Suzanne Collins',
 'Rules of Play: Game Design Fundamentals by Katie Salen',
 'The Player of Games by Iain M. Banks',
 'The Rain by Virginia Bergin ',
 'The Quillan Games by D.J. MacHale ',
 'Run for Cover by Eva Gray']

### RNN Sentiment Analysis

In [19]:
df_rnn = pd.read_csv("IMDB Dataset.csv")
df_rnn.drop_duplicates(inplace=True)

In [20]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

In [21]:
df_rnn["review"] = df_rnn["review"].str.lower()
df_rnn["review"] = df_rnn["review"].apply(remove_urls)
df_rnn["review"] = df_rnn["review"].apply(remove_punctuations)
df_rnn["review"] = df_rnn["review"].apply(remove_html)
df_rnn["review"] = df_rnn["review"].apply(remove_stopwords)
df_rnn["review"] = df_rnn["review"].apply(stemming)

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_rnn["sentiment"] = le.fit_transform(df_rnn["sentiment"])
y = df_rnn["sentiment"]

tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df_rnn["review"])

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)


In [25]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, no_of_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.no_of_layers = no_of_layers

        self.rnn = nn.RNN(input_size, hidden_size, no_of_layers, batch_first = True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.no_of_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:,-1,:])

        return out

In [26]:
input_size = X_train.shape[1]
rnn_model = RNN(input_size)

loss_criterion = nn.BCELoss()
optimizer = optim.Adam(rnn_model.parameters())

In [27]:
epochs = 10

for epoch in range(epochs):
    rnn_model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1)
        outputs = rnn_model(xb)
        outputs = torch.sigmoid(outputs.squeeze())

        loss = loss_criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item():.4f}")


epoch = 1/10 and loss = 0.3078
epoch = 2/10 and loss = 0.4103
epoch = 3/10 and loss = 0.2813
epoch = 4/10 and loss = 0.3075
epoch = 5/10 and loss = 0.2802
epoch = 6/10 and loss = 0.1998
epoch = 7/10 and loss = 0.2613
epoch = 8/10 and loss = 0.2447
epoch = 9/10 and loss = 0.1779
epoch = 10/10 and loss = 0.2381


In [28]:
rnn_model.eval()

with torch.no_grad():
    correct_otp = 0
    total_otp = 0
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        output = rnn_model(xb)
        predictions = (torch.sigmoid(output.squeeze()) > 0.5).float()

        correct_otp += (predictions == yb).sum().item()
        total_otp += yb.size(0)

    print(f"accuracy = {correct_otp/total_otp * 100}%")


accuracy = 85.80215791065847%


### Web Scraping For Reviews

In [90]:
import requests
import urllib.parse
from bs4 import BeautifulSoup

# header = {
#     "User-Agent": "Mozilla/5.0" ,
#     "Accept-Language" : "en-US,en;q=0.5"
# }

books_review_page_links = {}
books_reviews = {}
books_profile_reviews = {}
book_rating = {}
book_img = {}

In [91]:
def get_book_page(book_name_with_author):
    query = urllib.parse.quote(book_name_with_author)
    if books_review_page_links.get(book_name_with_author) == None or books_review_page_links.get(book_name_with_author) == []:
        URL = f"https://www.amazon.com/s?k={query}&i=stripbooks-intl-ship&crid=PLVOGG581S7X&sprefix=the+waves+%2Cstripbooks-intl-ship%2C514&ref=nb_sb_noss_2"
        header = {
            "User-Agent": "Mozilla/5.0" ,
            "referer" : URL,
            "Accept-Language" : "en-US,en;q=0.5"
        }
        response = requests.get(URL, headers = header)
        soup = BeautifulSoup(response.text, "lxml")
        link = soup.find("a", attrs = {
            "class" : "a-link-normal s-line-clamp-2 puis-line-clamp-3-for-col-4-and-8 s-link-style a-text-normal"
        })
        print(link)
        link = "https://www.amazon.com/" + link.get("href")
        books_review_page_links[book_name_with_author] = link

In [92]:
def get_review_page(book_name_with_author):
    if books_reviews.get(book_name_with_author) == None or books_reviews.get(book_name_with_author) == []:
        url_review_page = books_review_page_links[book_name_with_author]
        print(url_review_page)
        header = {
            "User-Agent": "Mozilla/5.0" ,
            "referer" : url_review_page,
            "Accept-Language" : "en-US,en;q=0.5"
        }
        response = requests.get(url_review_page, headers = header)
        soup = BeautifulSoup(response.text, "lxml")
        profiles = soup.find_all("div", attrs = {"class" : "a-row a-spacing-mini"})
        reviews = soup.find_all("div", attrs = {"class" : "a-expander-content reviewText review-text-content a-expander-partial-collapse-content"})
        img_link = soup.find("img", attrs = {"class" : "a-dynamic-image a-stretch-vertical media-block-image-tag"})
        book_img[book_name_with_author] = img_link.get('src') 

        # profile_with_review = {}
        # for i in range(len(reviews)):
        #     profile_with_review[profiles[i]] = reviews[i].text.split()
        # books_profile_reviews[book_name_with_author] = profile_with_review

        r = []
        for x in reviews:
            r.append(x.text.strip())
        books_reviews[book_name_with_author] = r

In [ ]:
book_name_with_author = "Mein Kampf by Adolf Hitler"

get_book_page(book_name_with_author)
get_review_page(book_name_with_author)

In [33]:
# Use SAME vectorizer
def reviews_to_stars(book_name):
    if book_rating.get(book_name) == None or book_rating.get(book_name) == None:
        reviews = books_reviews[book_name]
        if reviews:
            X_new = tf.transform(reviews)
            X_new = X_new.toarray()
            X_tensor = torch.tensor(X_new, dtype=torch.float32)

            with torch.no_grad():
                output = rnn_model(X_tensor.unsqueeze(1))  # (batch,1,5000)
                probs = torch.sigmoid(output)
            avg_score = probs.mean().item()
            return np.round(avg_score * 5, 2)
    else :
        return book_rating[book_name] 
    

### Connecting All models 

In [34]:
cleaned_df['title']

0                                 The Hunger Games
1        Harry Potter and the Order of the Phoenix
2                            To Kill a Mockingbird
3                              Pride and Prejudice
4                                         Twilight
                           ...                    
52473                                    Fractured
52474                                      Anasazi
52475                                       Marked
52476                                  Wayward Son
52477                          Daughter of Helaman
Name: title, Length: 42151, dtype: object

In [35]:
def book_recommendation_system(book_name):

    # 1. BOOK RECOMMENDATION
    recommended_books = recommend(book_name)

    # 2. REVIEW SCRAPING
    for book in recommended_books:
        get_book_page(book)
        get_review_page(book)

    # 3. SENTIMENT ANALYSIS
    for book in recommended_books:
        rating = reviews_to_stars(book)
        book_rating[book] = rating if rating != None else "Not Rated By Customers"
    
    return recommended_books, book_rating

In [ ]:
book_rating

In [ ]:
name = "To Kill a Mockingbird"
recommended_books, book_rating = book_recommendation_system(name)

In [ ]:
for book in recommended_books:
    print(book, ' -> ', book_img[book])

Magnolia by Kristi Cook   ->  https://m.media-amazon.com/images/I/51CR5iZGH9L._SY445_SX342_QL70_ML2_.jpg
Searching for Sylvie Lee by Jean Kwok   ->  https://m.media-amazon.com/images/I/51zCCBou7xL._SY445_SX342_QL70_ML2_.jpg
Moment of Truth by Kasie West   ->  https://m.media-amazon.com/images/I/51iV1xrr0oL._SY445_SX342_ML2_.jpg
The Last Forever by Deb Caletti   ->  https://m.media-amazon.com/images/I/51DTNu4wT1L._SY445_SX342_QL70_ML2_.jpg
Biggest Flirts by Jennifer Echols   ->  https://m.media-amazon.com/images/I/51avH3nUsdL._SY445_SX342_QL70_ML2_.jpg
Pieces of Us by Margie Gelbwasser   ->  https://m.media-amazon.com/images/I/41CQUgFyTXL._SY445_SX342_ML2_.jpg
If I'm Being Honest by Emily Wibberley   ->  https://m.media-amazon.com/images/I/511X07DCqvL._SY445_SX342_QL70_ML2_.jpg
Second Chance by Jane Green   ->  https://m.media-amazon.com/images/I/4137BS3GJ-S._SY445_SX342_QL70_ML2_.jpg
OCD Love Story by Corey Ann Haydu   ->  https://m.media-amazon.com/images/I/613rsfAcJRL._SY445_SX342_QL

### Saving Models 

In [39]:
import torch
import joblib
import pickle

torch.save(rnn_model.state_dict(), "model/rnn_model.pth")
joblib.dump(tf, "model/tfidf.pkl")
joblib.dump(cleaned_df, "model/books_df.pkl")
joblib.dump(vectors, "model/vectors.pkl")


def save_data():
    data = {
        "links": books_review_page_links,
        "reviews": books_reviews,
        "profile": books_profile_reviews,
        "rating": book_rating,
        "images": book_img
    }

    with open("data.pkl", "wb") as f:
        pickle.dump(data, f)